In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

In [0]:
df = spark.read.format('parquet').load('abfss://bronze@databrickse2esa.dfs.core.windows.net/products')
df = df.drop('_rescued_data')
display(df)


In [0]:
df.createOrReplaceTempView('products')

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_cata.bronze.discount_func(p_price DOUBLE)
RETURNS DOUBLE
LANGUAGE SQL
RETURN p_price*90/100

In [0]:
%sql
select *, databricks_cata.bronze.discount_func(price) as discounted_price 
from products 

In [0]:
df = df.withColumn('disocounted_price', expr("databricks_cata.bronze.discount_func(price)"))
display(df)

In [0]:
%sql
create or replace function databricks_cata.bronze.func_upper(p_val string )
returns string
language python
as 
$$ return p_val.upper() $$

In [0]:
%sql
select *, databricks_cata.bronze.func_upper(brand) from products

In [0]:
df.write.format('delta').mode('overwrite').save('abfss://silver@databrickse2esa.dfs.core.windows.net/products')

In [0]:
%sql
create table if not exists databricks_cata.silver.products_silver
using delta
location 'abfss://silver@databrickse2esa.dfs.core.windows.net/products';

select * from databricks_cata.silver.products_silver;